# VGG= Very Deep Convolutional Network (2014 Karen Simonyan and Andrew Zissermen in oxford VGG group)

Architecture:
- Input : $224*224$ image
- Conv kernel : $3*3$
- Max Pooling: $2*2$
- filters/channels increase as we go deeper


64 → 128 → 256 → 512 → 512

`VGG16`

13 conv layers + 3 fully connected layers
=16 layers


`VGG19`

16 conv layers + 3 fully connected layers
=19 layers

**why VGG**
- Much deeper than earlier models, more layers can learn more complex features.
- Small $3*3$ filters. Fewer parameters with more ReLU nonlinearities, Small filters + more layers = powerful feature learning with fewer parameters.

- Multiscale training: used different images scales helping it recognize features at different sizes.

```TEXT
architecutre is simple and consistent

Input
224 × 224 × 3
      ↓
┌──────────────────────┐
│ Conv 3×3, 64 filters │
│ Conv 3×3, 64 filters │
│ MaxPool 2×2          │
└──────────────────────┘
      ↓
┌───────────────────────┐
│ Conv 3×3, 128 filters │
│ Conv 3×3, 128 filters │
│ MaxPool 2×2           │
└───────────────────────┘
      ↓
┌───────────────────────┐
│ Conv 3×3, 256 filters │
│ Conv 3×3, 256 filters │
│ Conv 3×3, 256 filters │
│ MaxPool 2×2           │
└───────────────────────┘
      ↓
┌───────────────────────┐
│ Conv 3×3, 512 filters │
│ Conv 3×3, 512 filters │
│ Conv 3×3, 512 filters │
│ MaxPool 2×2           │
└───────────────────────┘
      ↓
┌───────────────────────┐
│ Conv 3×3, 512 filters │
│ Conv 3×3, 512 filters │
│ Conv 3×3, 512 filters │
│ MaxPool 2×2           │
└───────────────────────┘
      ↓
    Flatten
      ↓
 Fully Connected
      ↓
 Fully Connected
      ↓
 Fully Connected
      ↓
 Classification

```

VGG uses many 3×3 filters, gradually increases filters from 64 → 512, uses pooling to shrink the image, then Flatten + 3 Dense/FC layers for classification

- CNN:  32 → 64 filters
- VGG:  64 → 128 → 256 → 512 → 512 filters

In [ ]:
!pip install torch


In [15]:
import torch.nn as nn


class VGG19(nn.Module):
    def __init__(self, num_classes=1000):
        super(VGG19, self).__init__()

        # ==========================================================
        # FEATURE EXTRACTOR
        # ==========================================================
        # This part extracts important features from the input image
        # using convolutional layers, ReLU activation, and max pooling.
        #
        # Input image:
        #   3 channels (RGB)
        #
        # VGG19 gradually increases the number of channels:
        #   3 -> 64 -> 128 -> 256 -> 512
        #
        # At the same time, max pooling reduces the spatial dimensions.
        # ==========================================================

        self.feature_extractor = nn.Sequential(

            # -------------------- Block 1 --------------------
            # Input:  3 channels (RGB)
            # Output: 64 feature maps
            # Kernel: 3x3
            # Padding: 1 keeps height and width unchanged
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            # 64 input channels -> 64 output channels
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            # Reduces height and width by a factor of 2
            # Example: 224x224 -> 112x112
            nn.MaxPool2d(kernel_size=2, stride=2),


            # -------------------- Block 2 --------------------
            # 64 feature maps -> 128 feature maps
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),

            # Spatial dimensions are reduced by half
            nn.MaxPool2d(kernel_size=2, stride=2),


            # -------------------- Block 3 --------------------
            # 128 feature maps -> 256 feature maps
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),

            # Reduce spatial dimensions
            nn.MaxPool2d(kernel_size=2, stride=2),


            # -------------------- Block 4 --------------------
            # 256 feature maps -> 512 feature maps
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),

            # Reduce spatial dimensions
            nn.MaxPool2d(kernel_size=2, stride=2),


            # -------------------- Block 5 --------------------
            # Keep 512 feature maps
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),

            # Final max pooling
            nn.MaxPool2d(kernel_size=2, stride=2),
        )


        # ==========================================================
        # ADAPTIVE AVERAGE POOLING
        # ==========================================================
        # Converts the spatial dimensions of the feature maps
        # to exactly 7 x 7.
        #
        # Regardless of the spatial size coming from the feature
        # extractor, the output will be:
        #
        #   512 x 7 x 7
        #
        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(7, 7))


        # ==========================================================
        # CLASSIFIER
        # ==========================================================
        # This part takes the extracted features and performs
        # classification.
        #
        # 512 x 7 x 7 = 25088 features
        #
        # These are flattened and passed through fully connected
        # (Linear) layers.
        # ==========================================================

        self.classifier = nn.Sequential(

            # Input:
            #   512 * 7 * 7 = 25088 features
            #
            # Output:
            #   4096 features
            nn.Linear(512 * 7 * 7, 4096),

            # Activation function
            nn.ReLU(),

            # Randomly disables 50% of neurons during training
            # to help prevent overfitting
            nn.Dropout(0.5),

            # Second fully connected layer
            nn.Linear(4096, 4096),
            nn.ReLU(),

            # Another dropout layer
            nn.Dropout(0.5),

            # Final classification layer
            #
            # 4096 -> number of classes
            #
            # For ImageNet:
            # num_classes = 1000
            nn.Linear(4096, num_classes)
        )


    # ==========================================================
    # FORWARD PASS
    # ==========================================================
    def forward(self, x):

        # Pass input image through convolutional feature extractor
        x = self.feature_extractor(x)

        # Convert feature maps to 7x7 spatial dimensions
        x = self.avgpool(x)

        # Flatten the tensor
        #
        # Example:
        # [batch_size, 512, 7, 7]
        #
        # becomes:
        # [batch_size, 25088]
        x = x.view(x.size(0), -1)

        # Pass flattened features through classifier
        x = self.classifier(x)

        # Return class scores
        return x
